# Formation of UPX Metadata

In [1]:
# importing all libraries
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display


In [2]:
# reading the speaker information file and making a dataframe
upx_info_df=pd.read_csv("UPX_info.csv")
upx_info_df.head()

,SPEAKER-ID,GENDER,AGE-Y,AGE-M,AGE,SSD SUBTYPE
0,01F,Female,8,8,8.67,inconsistent phonological disorder
1,02F,Female,7,8,7.67,phonological disorder
2,03F,Female,10,11,10.92,childhood apraxia of speech
3,04M,Male,7,2,7.17,phonological delay
4,05M,Male,6,5,6.42,vowel disorder


In [3]:
# checking the shape of dataset
upx_info_df.shape

(20, 6)

In [4]:
# checking for any missing values
upx_info_df.isna().sum()

SPEAKER-ID     0
GENDER         0
AGE-Y          0
AGE-M          0
AGE            0
SSD SUBTYPE    0
dtype: int64

In [5]:
# now forming a metatable of the data based inside core-upx directory

dataset = "upx" # storing dataset name as a string
dataset_root = Path("core-upx") # storing the folder path name

# sub-folders inside the root folder
core_dir = dataset_root/ "core"
lab_dir = dataset_root/ "speaker_labels"/ "lab"

rows = []

# now looping through dataset

# looping through each speaker folder
for speaker_dir in sorted(core_dir.iterdir()): # iterate is used to make a iterative generator for the for loop because core_dir is just a path
    if not speaker_dir.is_dir(): # check for folder only
        continue # skip if not found

    speaker_id = speaker_dir.name # taking the folder name

    for session_dir in sorted(speaker_dir.iterdir()):
        if not session_dir.is_dir(): # check for folder only
            continue

        session_id = session_dir.name

        for wav_path in sorted(session_dir.glob("*.wav")):
            file_id = wav_path.stem # skip the extension part

            txt_path = session_dir/ f"{file_id}.txt"
            full_id = f"{speaker_id}-{session_id}-{file_id}" # pattern to match the text file format
            lab_path = lab_dir/ f"{full_id}.lab"

            rows.append({
                "dataset_name": dataset,
                "speaker_id": speaker_id,
                "session_id": session_id,
                "file_id" : file_id, 
                "full_id": full_id,
                "raw_wav_path": str(wav_path),
                "txt_path": str(txt_path) if txt_path.exists() else None,
                "speaker_label_path": str(lab_path) if lab_path.exists() else None,
                "class_label": 1
            })

upx_metadata=pd.DataFrame(rows)
upx_metadata.head()


,dataset_name,speaker_id,session_id,file_id,full_id,raw_wav_path,txt_path,speaker_label_path,class_label
0,upx,01F,BL1,001E,01F-BL1-001E,core-upx/core/01F/BL1/001E.wav,core-upx/core/01F/BL1/001E.txt,core-upx/speaker_labels/lab/01F-BL1-001E.lab,1
1,upx,01F,BL1,002E,01F-BL1-002E,core-upx/core/01F/BL1/002E.wav,core-upx/core/01F/BL1/002E.txt,core-upx/speaker_labels/lab/01F-BL1-002E.lab,1
2,upx,01F,BL1,003A,01F-BL1-003A,core-upx/core/01F/BL1/003A.wav,core-upx/core/01F/BL1/003A.txt,core-upx/speaker_labels/lab/01F-BL1-003A.lab,1
3,upx,01F,BL1,004A,01F-BL1-004A,core-upx/core/01F/BL1/004A.wav,core-upx/core/01F/BL1/004A.txt,core-upx/speaker_labels/lab/01F-BL1-004A.lab,1
4,upx,01F,BL1,005A,01F-BL1-005A,core-upx/core/01F/BL1/005A.wav,core-upx/core/01F/BL1/005A.txt,core-upx/speaker_labels/lab/01F-BL1-005A.lab,1


In [6]:
# checking for null values
upx_metadata.isna().sum()

dataset_name           0
speaker_id             0
session_id             0
file_id                0
full_id                0
raw_wav_path           0
txt_path               0
speaker_label_path    17
class_label            0
dtype: int64

In [7]:
# checking for column names
print(upx_info_df.columns)
print(upx_metadata.columns)

Index(['SPEAKER-ID', 'GENDER', 'AGE-Y', 'AGE-M', 'AGE', 'SSD SUBTYPE'], dtype='str')
Index(['dataset_name', 'speaker_id', 'session_id', 'file_id', 'full_id',
       'raw_wav_path', 'txt_path', 'speaker_label_path', 'class_label'],
      dtype='str')


In [8]:
# renaming the speaker_id
upx_info_df=upx_info_df.rename(columns={"SPEAKER-ID":"speaker_id"})

In [9]:
print(upx_info_df.columns)
print(upx_metadata.columns)

Index(['speaker_id', 'GENDER', 'AGE-Y', 'AGE-M', 'AGE', 'SSD SUBTYPE'], dtype='str')
Index(['dataset_name', 'speaker_id', 'session_id', 'file_id', 'full_id',
       'raw_wav_path', 'txt_path', 'speaker_label_path', 'class_label'],
      dtype='str')


In [10]:
# merging the info data with the metadata
upx_metadata=upx_metadata.merge(
    upx_info_df,
    on = "speaker_id",
    how = "left")

In [11]:
upx_metadata.head()

,dataset_name,speaker_id,session_id,file_id,full_id,raw_wav_path,txt_path,speaker_label_path,class_label,GENDER,AGE-Y,AGE-M,AGE,SSD SUBTYPE
0,upx,01F,BL1,001E,01F-BL1-001E,core-upx/core/01F/BL1/001E.wav,core-upx/core/01F/BL1/001E.txt,core-upx/speaker_labels/lab/01F-BL1-001E.lab,1,Female,8,8,8.67,inconsistent phonological disorder
1,upx,01F,BL1,002E,01F-BL1-002E,core-upx/core/01F/BL1/002E.wav,core-upx/core/01F/BL1/002E.txt,core-upx/speaker_labels/lab/01F-BL1-002E.lab,1,Female,8,8,8.67,inconsistent phonological disorder
2,upx,01F,BL1,003A,01F-BL1-003A,core-upx/core/01F/BL1/003A.wav,core-upx/core/01F/BL1/003A.txt,core-upx/speaker_labels/lab/01F-BL1-003A.lab,1,Female,8,8,8.67,inconsistent phonological disorder
3,upx,01F,BL1,004A,01F-BL1-004A,core-upx/core/01F/BL1/004A.wav,core-upx/core/01F/BL1/004A.txt,core-upx/speaker_labels/lab/01F-BL1-004A.lab,1,Female,8,8,8.67,inconsistent phonological disorder
4,upx,01F,BL1,005A,01F-BL1-005A,core-upx/core/01F/BL1/005A.wav,core-upx/core/01F/BL1/005A.txt,core-upx/speaker_labels/lab/01F-BL1-005A.lab,1,Female,8,8,8.67,inconsistent phonological disorder


In [12]:
# rearranging the columns
upx_metadata = upx_metadata[
    [
        "dataset_name",
        "speaker_id",
        "session_id",
        "file_id",
        "GENDER",
        "AGE-Y",
        "AGE-M",
        "AGE",
        "SSD SUBTYPE",
        "full_id",
        "raw_wav_path",
        "txt_path",
        "speaker_label_path",
        "class_label"
    ]
]
upx_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,SSD SUBTYPE,full_id,raw_wav_path,txt_path,speaker_label_path,class_label
0,upx,01F,BL1,001E,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-001E,core-upx/core/01F/BL1/001E.wav,core-upx/core/01F/BL1/001E.txt,core-upx/speaker_labels/lab/01F-BL1-001E.lab,1
1,upx,01F,BL1,002E,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-002E,core-upx/core/01F/BL1/002E.wav,core-upx/core/01F/BL1/002E.txt,core-upx/speaker_labels/lab/01F-BL1-002E.lab,1
2,upx,01F,BL1,003A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-003A,core-upx/core/01F/BL1/003A.wav,core-upx/core/01F/BL1/003A.txt,core-upx/speaker_labels/lab/01F-BL1-003A.lab,1
3,upx,01F,BL1,004A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-004A,core-upx/core/01F/BL1/004A.wav,core-upx/core/01F/BL1/004A.txt,core-upx/speaker_labels/lab/01F-BL1-004A.lab,1
4,upx,01F,BL1,005A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-005A,core-upx/core/01F/BL1/005A.wav,core-upx/core/01F/BL1/005A.txt,core-upx/speaker_labels/lab/01F-BL1-005A.lab,1


In [13]:
# checking the missing values again for confirmation
upx_metadata.isna().sum()

dataset_name           0
speaker_id             0
session_id             0
file_id                0
GENDER                 0
AGE-Y                  0
AGE-M                  0
AGE                    0
SSD SUBTYPE            0
full_id                0
raw_wav_path           0
txt_path               0
speaker_label_path    17
class_label            0
dtype: int64

## Reading the prompt text for txt file

The prompt text will allow us to filter the recordings in pre-processing since some of the recordings are blank or contain irregular text information, which is not related with the actual children speeches.

In [14]:
# reading text from the text files
def read_prompt_text(txt_path):
    if pd.isna(txt_path):
        return None
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            return f.readline().strip()
    except Exception:
        return None

upx_metadata["prompt_text"] = upx_metadata["txt_path"].apply(read_prompt_text) # function will apply on each value on the column
upx_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,SSD SUBTYPE,full_id,raw_wav_path,txt_path,speaker_label_path,class_label,prompt_text
0,upx,01F,BL1,001E,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-001E,core-upx/core/01F/BL1/001E.wav,core-upx/core/01F/BL1/001E.txt,core-upx/speaker_labels/lab/01F-BL1-001E.lab,1,Swallow
1,upx,01F,BL1,002E,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-002E,core-upx/core/01F/BL1/002E.wav,core-upx/core/01F/BL1/002E.txt,core-upx/speaker_labels/lab/01F-BL1-002E.lab,1,Swallow
2,upx,01F,BL1,003A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-003A,core-upx/core/01F/BL1/003A.wav,core-upx/core/01F/BL1/003A.txt,core-upx/speaker_labels/lab/01F-BL1-003A.lab,1,elephant umbrella train swing
3,upx,01F,BL1,004A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-004A,core-upx/core/01F/BL1/004A.wav,core-upx/core/01F/BL1/004A.txt,core-upx/speaker_labels/lab/01F-BL1-004A.lab,1,bread duck giraffe five
4,upx,01F,BL1,005A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-005A,core-upx/core/01F/BL1/005A.wav,core-upx/core/01F/BL1/005A.txt,core-upx/speaker_labels/lab/01F-BL1-005A.lab,1,teeth watch orange school


## Audio Segmentation using speaker_label_path

Audio Segmentation is performed using available speaker labels timestamps for child speech part inside the original raw recordings.


In [15]:
# helper function to perform the segmentation
def diarize_with_speaker_labels(wav_path, lab_path, output_path,
                                keep_label="CHILD",
                                margin_ms=20, # margin to remove the cross boundry issue between child and therapist speech
                                min_seg_ms=100): # to filter the recordings having duration less than 100 milli seconds
   
    audio, sr = sf.read(wav_path)

    # converting stereo to mono channel
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    margin_samples = round(sr * margin_ms / 1000)
    min_seg_samples = round(sr * min_seg_ms / 1000)
    target_label = keep_label.upper()

    kept_segments = []
    found_child = False
    found_slt = False
    has_target_label = False
    n_target_segments_raw = 0

    with open(lab_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3:
                continue

            start_htk = int(parts[0])
            end_htk = int(parts[1])
            label = parts[2].strip().upper()

            if label == "CHILD":
                found_child = True
            if label == "SLT":
                found_slt = True

            if label == target_label:
                has_target_label = True
                n_target_segments_raw += 1

                start_sample = round(start_htk * sr / 10_000_000)
                end_sample = round(end_htk * sr / 10_000_000)

                # trim boundaries slightly
                start_sample += margin_samples
                end_sample -= margin_samples

                # clip bounds
                start_sample = max(0, start_sample)
                end_sample = min(len(audio), end_sample)

                if end_sample > start_sample:
                    seg = audio[start_sample:end_sample]
                    if len(seg) >= min_seg_samples:
                        kept_segments.append(seg)

    raw_duration_sec = len(audio) / sr

    if not kept_segments:
        return {
            "status": "no_target_labels" if not has_target_label else "no_target_segments_after_filters",
            "target_label": target_label,
            "sample_rate": sr,
            "raw_duration_sec": raw_duration_sec,
            "child_duration_sec": 0.0,
            "n_child_segments": 0,
            "has_target_label": has_target_label,
            "n_target_segments_raw": n_target_segments_raw,
            "n_target_segments_kept": 0,
            "has_child": found_child,
            "has_slt": found_slt
        }

    child_audio = np.concatenate(kept_segments)
    sf.write(output_path, child_audio, sr)

    return {
        "status": "processed",
        "target_label": target_label,
        "sample_rate": sr,
        "raw_duration_sec": raw_duration_sec,
        "child_duration_sec": len(child_audio) / sr,
        "n_child_segments": len(kept_segments),
        "has_target_label": has_target_label,
        "n_target_segments_raw": n_target_segments_raw,
        "n_target_segments_kept": len(kept_segments),
        "has_child": found_child,
        "has_slt": found_slt
    }

In [16]:
# storing the processed recordings after segmentation
row = upx_metadata.iloc[0]

output_dir = Path("upx_processed_child_wav")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / f"{row['full_id']}_child.wav"

result = diarize_with_speaker_labels(
    wav_path=row["raw_wav_path"],
    lab_path=row["speaker_label_path"],
    output_path=output_path
)

print(result)
print("Saved to:", output_path)

{'status': 'processed', 'target_label': 'CHILD', 'sample_rate': 22050, 'raw_duration_sec': 7.848344671201814, 'child_duration_sec': 1.32, 'n_child_segments': 1, 'has_target_label': True, 'n_target_segments_raw': 1, 'n_target_segments_kept': 1, 'has_child': True, 'has_slt': True}
Saved to: upx_processed_child_wav/01F-BL1-001E_child.wav


In [17]:
upx_metadata.loc[0, "processed_child_wav_path"] = str(output_path)
upx_metadata.loc[0, "raw_duration_sec"] = result["raw_duration_sec"]
upx_metadata.loc[0, "child_duration_sec"] = result["child_duration_sec"]
upx_metadata.loc[0, "n_child_segments"] = result["n_child_segments"]
upx_metadata.loc[0, "has_child"] = result["has_child"]
upx_metadata.loc[0, "has_slt"] = result["has_slt"]
upx_metadata.loc[0, "status"] = result["status"]
upx_metadata.loc[0, "diarization_method"] = "provided_speaker_labels"

In [18]:
# testing before and after recording for first row
print("Before:")
display(Audio(row["raw_wav_path"]))

print("After:")
Audio(row["raw_wav_path"])
Audio(str(output_path))

Before:


After:


In [19]:
upx_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,SSD SUBTYPE,full_id,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
0,upx,01F,BL1,001E,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-001E,...,1,Swallow,upx_processed_child_wav/01F-BL1-001E_child.wav,7.848345,1.32,1.0,True,True,processed,provided_speaker_labels
1,upx,01F,BL1,002E,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-002E,...,1,Swallow,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,upx,01F,BL1,003A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-003A,...,1,elephant umbrella train swing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,upx,01F,BL1,004A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-004A,...,1,bread duck giraffe five,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,upx,01F,BL1,005A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-005A,...,1,teeth watch orange school,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Now applying segmentation on whole data frame

In [20]:
output_dir = Path("upx_processed_child_wav")
output_dir.mkdir(exist_ok=True)

for idx, row in upx_metadata.iterrows():
    wav_path = row["raw_wav_path"]
    lab_path = row["speaker_label_path"]

    if pd.isna(wav_path) or pd.isna(lab_path):
        upx_metadata.loc[idx, "status"] = "missing_path"
        continue

    output_path = output_dir / f"{row['full_id']}_child.wav"

    try:
        result = diarize_with_speaker_labels(
            wav_path=wav_path,
            lab_path=lab_path,
            output_path=output_path
        )

        upx_metadata.loc[idx, "processed_child_wav_path"] = str(output_path)
        upx_metadata.loc[idx, "raw_duration_sec"] = result["raw_duration_sec"]
        upx_metadata.loc[idx, "child_duration_sec"] = result["child_duration_sec"]
        upx_metadata.loc[idx, "n_child_segments"] = result["n_child_segments"]
        upx_metadata.loc[idx, "has_child"] = result["has_child"]
        upx_metadata.loc[idx, "has_slt"] = result["has_slt"]
        upx_metadata.loc[idx, "status"] = result["status"]
        upx_metadata.loc[idx, "diarization_method"] = "provided_speaker_labels"

    except Exception as e:
        upx_metadata.loc[idx, "status"] = f"error: {e}"

In [21]:
# checking the data again
upx_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,SSD SUBTYPE,full_id,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
0,upx,01F,BL1,001E,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-001E,...,1,Swallow,upx_processed_child_wav/01F-BL1-001E_child.wav,7.848345,1.320000,1.0,True,True,processed,provided_speaker_labels
1,upx,01F,BL1,002E,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-002E,...,1,Swallow,upx_processed_child_wav/01F-BL1-002E_child.wav,12.074376,2.480045,3.0,True,True,processed,provided_speaker_labels
2,upx,01F,BL1,003A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-003A,...,1,elephant umbrella train swing,upx_processed_child_wav/01F-BL1-003A_child.wav,9.659501,4.090023,4.0,True,False,processed,provided_speaker_labels
3,upx,01F,BL1,004A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-004A,...,1,bread duck giraffe five,upx_processed_child_wav/01F-BL1-004A_child.wav,11.470658,4.560000,6.0,True,False,processed,provided_speaker_labels
4,upx,01F,BL1,005A,Female,8,8,8.67,inconsistent phonological disorder,01F-BL1-005A,...,1,teeth watch orange school,upx_processed_child_wav/01F-BL1-005A_child.wav,11.470658,5.430023,4.0,True,False,processed,provided_speaker_labels


In [22]:
row1=upx_metadata.iloc[1]
Audio(filename=row1["raw_wav_path"])


In [23]:
row1=upx_metadata.iloc[1]
Audio(filename=row1["processed_child_wav_path"])

In [24]:
# testing row index 15 before segmentation
row=upx_metadata.iloc[15]
Audio(filename=row["raw_wav_path"])

In [25]:
# testing row index 15 before segmentation
row=upx_metadata.iloc[15]
Audio(filename=row["processed_child_wav_path"])

In [26]:
# checking the null values again
upx_metadata.isna().sum()

dataset_name                 0
speaker_id                   0
session_id                   0
file_id                      0
GENDER                       0
AGE-Y                        0
AGE-M                        0
AGE                          0
SSD SUBTYPE                  0
full_id                      0
raw_wav_path                 0
txt_path                     0
speaker_label_path          17
class_label                  0
prompt_text                  0
processed_child_wav_path    17
raw_duration_sec            17
child_duration_sec          17
n_child_segments            17
has_child                   17
has_slt                     17
status                       0
diarization_method          17
dtype: int64

In [27]:
upx_metadata[upx_metadata["speaker_label_path"].isna()]

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,SSD SUBTYPE,full_id,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
959,upx,07M,BL2,055D,Male,8,11,8.92,childhood apraxia of speech,07M-BL2-055D,...,1,ptk-1SD,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
962,upx,07M,BL2,058D,Male,8,11,8.92,childhood apraxia of speech,07M-BL2-058D,...,1,ptk+2SD,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
1014,upx,07M,BL4,001E,Male,8,11,8.92,childhood apraxia of speech,07M-BL4-001E,...,1,Swallow,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
1106,upx,08M,BL1,044E,Male,10,2,10.17,childhood apraxia of speech,08M-BL1-044E,...,1,Swallow,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
1346,upx,10M,BL2,001E,Male,13,4,13.33,childhood apraxia of speech,10M-BL2-001E,...,1,Swallow,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
1399,upx,10M,BL2,054E,Male,13,4,13.33,childhood apraxia of speech,10M-BL2-054E,...,1,Swallow,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
1444,upx,10M,BL3,045E,Male,13,4,13.33,childhood apraxia of speech,10M-BL3-045E,...,1,Swallow,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
1481,upx,10M,BL4,037E,Male,13,4,13.33,childhood apraxia of speech,10M-BL4-037E,...,1,Swallow,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
1553,upx,12M,BL1,001E,Male,6,3,6.25,phonological delay,12M-BL1-001E,...,1,Swallow,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
1664,upx,12M,BL3,001E,Male,6,3,6.25,phonological delay,12M-BL3-001E,...,1,Swallow,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN


### About 17 rows have missing child labels, which means that their segmentation is not possible based on our approach. So we are dropping these instances.

In [28]:
# dropping missing child labels instances
upx_metadata.dropna(axis=0, inplace=True)

In [29]:
# checking for empty rows again.
upx_metadata.isna().sum()

dataset_name                0
speaker_id                  0
session_id                  0
file_id                     0
GENDER                      0
AGE-Y                       0
AGE-M                       0
AGE                         0
SSD SUBTYPE                 0
full_id                     0
raw_wav_path                0
txt_path                    0
speaker_label_path          0
class_label                 0
prompt_text                 0
processed_child_wav_path    0
raw_duration_sec            0
child_duration_sec          0
n_child_segments            0
has_child                   0
has_slt                     0
status                      0
diarization_method          0
dtype: int64

In [30]:
# checking all values inside the column "has_child"
upx_metadata["has_child"].unique()

array([True, False], dtype=object)

In [31]:
# checking for false values rows
upx_metadata[upx_metadata["has_child"] == False]

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,SSD SUBTYPE,full_id,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
326,upx,03F,BL2,001E,Female,10,11,10.92,childhood apraxia of speech,03F-BL2-001E,...,1,Swallow,upx_processed_child_wav/03F-BL2-001E_child.wav,7.848345,0.0,0.0,False,True,no_target_labels,provided_speaker_labels
376,upx,03F,BL3,001E,Female,10,11,10.92,childhood apraxia of speech,03F-BL3-001E,...,1,Swallow,upx_processed_child_wav/03F-BL3-001E_child.wav,10.263220,0.0,0.0,False,True,no_target_labels,provided_speaker_labels
473,upx,04M,BL1,001E,Male,7,2,7.17,phonological delay,04M-BL1-001E,...,1,Swallow,upx_processed_child_wav/04M-BL1-001E_child.wav,5.433469,0.0,0.0,False,True,no_target_labels,provided_speaker_labels
1013,upx,07M,BL3,051E,Male,8,11,8.92,childhood apraxia of speech,07M-BL3-051E,...,1,Swallow,upx_processed_child_wav/07M-BL3-051E_child.wav,10.263220,0.0,0.0,False,True,no_target_labels,provided_speaker_labels
1062,upx,07M,BL5,024E,Male,8,11,8.92,childhood apraxia of speech,07M-BL5-024E,...,1,Swallow,upx_processed_child_wav/07M-BL5-024E_child.wav,3.622313,0.0,0.0,False,False,no_target_labels,provided_speaker_labels
1196,upx,08M,BL3,060E,Male,10,2,10.17,childhood apraxia of speech,08M-BL3-060E,...,1,Swallow,upx_processed_child_wav/08M-BL3-060E_child.wav,4.226032,0.0,0.0,False,True,no_target_labels,provided_speaker_labels
1230,upx,09M,BL2,001E,Male,7,9,7.75,phonological disorder,09M-BL2-001E,...,1,Swallow,upx_processed_child_wav/09M-BL2-001E_child.wav,8.452063,0.0,0.0,False,True,no_target_labels,provided_speaker_labels
1345,upx,10M,BL1,048E,Male,13,4,13.33,childhood apraxia of speech,10M-BL1-048E,...,1,Swallow,upx_processed_child_wav/10M-BL1-048E_child.wav,7.848345,0.0,0.0,False,False,no_target_labels,provided_speaker_labels
1445,upx,10M,BL4,001E,Male,13,4,13.33,childhood apraxia of speech,10M-BL4-001E,...,1,Swallow,upx_processed_child_wav/10M-BL4-001E_child.wav,6.037188,0.0,0.0,False,False,no_target_labels,provided_speaker_labels
1855,upx,13M,BL3,036E,Male,7,4,7.33,inconsistent phonological disorder,13M-BL3-036E,...,1,Swallow,upx_processed_child_wav/13M-BL3-036E_child.wav,7.848345,0.0,0.0,False,True,no_target_labels,provided_speaker_labels


### So there are about 17 more records which does not have child labels. So we need to remove them as well.

In [32]:
upx_metadata.drop(upx_metadata[upx_metadata["has_child"] == False].index, inplace=True)

In [33]:
# checking the values again
upx_metadata["has_child"].unique()


array([True], dtype=object)

In [34]:
upx_metadata["prompt_text"].unique()

<ArrowStringArray>
[                           'Swallow',      'elephant umbrella train swing',
            'bread duck giraffe five',          'teeth watch orange school',
 'crab biscuits thank-you helicopter',              'egg splash square pig',
            'gloves queen three frog',       'yellow strawberry spider web',
           'sheep snake pram feather',     'tomato monkey toothbrush apple',
 ...
           'zombie nose zone whistle',                 'bays puss size sip',
             'seep glasses xylophone',               'sucks sore zoo shoot',
            'hoist suit sheep shower',               'smoke speed ship fez',
                              's s s',                         'ees os ahs',
                           'sh sh sh',                        'she sho sha']
Length: 450, dtype: str

In [35]:
upx_metadata.shape

(2758, 23)

In [36]:
# checking for prompt text
vc = upx_metadata["prompt_text"].value_counts(dropna=False)
print(vc.to_string())

prompt_text
Swallow                                                            91
down link pat get                                                  28
Kenny drank a tiny tin of coke                                     28
Ken likes scones with cream and apricot jam                        27
My Granny Maggie got a golden gown                                 26
Baby Gary's got a bag of lego                                      26
scrawny tube toot cream                                            25
smokey meet lit broccoli                                           25
team gap grovel tip                                                25
deer buggy debt crow                                               25
avocado KitKat digger dumb                                         25
dawn fig garden tab                                                25
Coca-Cola pack bog bed                                             25
tool hockey Maggie gone                                            25
gift tie

#### So some prompt text is not relevant, which shows that we need to remove these instances as well.

In [37]:
col = "prompt_text"

#  Regex for rows to remove
score_code_pat = r"^(?:ptk|tk|p|t|k)(?:[+-]\d+SD|-mean)$"
task_label_pat = r"^Connected speech(?: picture)?\s*\d+$"

remove_pat = re.compile(
    f"(?:{score_code_pat})|(?:{task_label_pat})",
    flags=re.IGNORECASE
)

# building mask
s = upx_metadata[col].fillna("").astype(str).str.strip()
mask_remove = s.str.match(remove_pat, na=False)

# now getting clean dataframe
upx_metadata_clean = upx_metadata.loc[~mask_remove].copy()

print("Before:", len(upx_metadata))
print("Removed:", mask_remove.sum())
print("After:", len(upx_metadata_clean))

Before: 2758
Removed: 243
After: 2515


In [38]:
# storing the clean upx_metadata csv file
upx_metadata_clean.to_csv("upx_metadata_clean.csv", index=False)